# DATA VORTEX A'26 — Round 3 Analysis Notebook

**Topic:** Public Reaction to a Viral Product Launch  
**Data Source:** Bluesky Social  
**Analysis Date:** 2026-09-22  

This notebook reproduces the Round 3 workflow. All results are loaded from the final output CSVs produced by the pipeline scripts. The frozen Round 2 model was not retrained or modified.

**Final dataset stats:**
- Original collected posts: 1,320
- Relevant posts (after topic-relevance filtering): **587**
- Sentiment shifts detected: **10**
- Engagement spikes detected: **1**
- Total flagged events: **11**
- Trigger evidence: 0 Strong / 1 Weak / 10 Unexplained

## 0. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

BASE = Path('../')  # datavortex root
R3 = BASE / 'Round3'
print('Paths set. Base:', BASE.resolve())

## 1. Data Collection

Data was collected from Bluesky Social via `Round3/src/01_collect_bluesky.py`.
- Authenticated via `bsky.social` (credentials in `Round3/.env`, not tracked by git)
- **Single-shot collection** — one API run on 2026-09-22
- Queries: `Apple iPhone`, `iphone`, `technology`
- Deduplication by post URI

**Fix 1 (relevance filtering):** 733 off-topic posts removed. Original raw data preserved unchanged.

In [ ]:
raw_df = pd.read_csv(R3 / 'data/raw/bluesky_raw_final.csv')
rel_df = pd.read_csv(R3 / 'data/raw/bluesky_relevant.csv')
excl_df = pd.read_csv(R3 / 'data/raw/exclusion_log.csv')

print(f'Original posts collected : {len(raw_df)}')
print(f'Posts after relevance filter: {len(rel_df)}')
print(f'Excluded posts           : {len(excl_df)}')
print()
print('Query term distribution (original):')
print(raw_df['query_term'].value_counts())

## 2. Preprocessing & Sentiment Inference

The exact Round 2 `clean_text()` function and frozen TF-IDF + LinearSVC model were applied via `Round3/src/02_process_and_predict.py`. No retraining occurred.

- **Round 2 Val Macro-F1:** 60.11%
- **Round 2 Test Macro-F1:** 60.93%
- Classes: Negative, Neutral, Positive
- `future_dated_anomaly` flag added for posts where `created_at > collected_at`

In [ ]:
enriched_df = pd.read_csv(R3 / 'data/processed/bluesky_enriched.csv')

print(f'Enriched posts: {len(enriched_df)}')
print(f'Future-dated anomalies: {(enriched_df["future_dated_anomaly"] == True).sum()}')
print()
print('Sentiment distribution:')
print(enriched_df['sentiment_label'].value_counts())
print()
print(enriched_df['sentiment_label'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

## 3. Timeline Construction

Built by `Round3/src/03_build_timeline.py`.
- 22 hourly UTC buckets
- Engagement = likes + reposts + replies + quotes per bucket
- Original `created_at` timestamps preserved unaltered (6 future-dated anomalies included)

In [ ]:
timeline_df = pd.read_csv(R3 / 'data/processed/hourly_timeline.csv')

print(f'Hourly buckets: {len(timeline_df)}')
print(f'Total engagement across all buckets: {timeline_df["total_engagement"].sum()}')
print()
display(timeline_df[['hour', 'post_count', 'total_engagement', 'avg_engagement',
                      'pct_negative', 'pct_neutral', 'pct_positive']].to_string(index=False))

## 4. Event Detection

Built by `Round3/src/04_detect_events.py`.

**Fix 2:** Engagement spike detection corrected to use `total_engagement` (likes+reposts+replies+quotes), not post count.

- Sentiment shift threshold: ≥15 pp change in polarity (`pct_positive − pct_negative`)
- Engagement spike threshold: `mean_engagement + 2σ` = **115.21 engagement units**
- All 22 buckets scanned

In [ ]:
events_df = pd.read_csv(R3 / 'data/processed/flagged_events.csv')

print(f'Total flagged events: {len(events_df)}')
print(f'Sentiment shifts:     {len(events_df[events_df["event_type"] == "sentiment_shift"])}')
print(f'Engagement spikes:    {len(events_df[events_df["event_type"] == "engagement_spike"])}')
print()
display(events_df[['event_id', 'timestamp', 'event_type', 'metric_value',
                   'delta_or_zscore', 'threshold', 'polarity']].to_string(index=False))

## 5. Figure 1 — Sentiment & Activity Timeline

Generated by `Round3/src/07_build_report_figures.py`.

In [ ]:
fig1 = mpimg.imread(str(R3 / 'reports/figure1_timeline.png'))
plt.figure(figsize=(14, 6))
plt.imshow(fig1)
plt.axis('off')
plt.title('Figure 1: Sentiment & Activity Timeline', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Topic & Entity Analysis

Built by `Round3/src/05_topic_entity_analysis.py`.

**Fix 3:**
- `<url>` and HTTP artifacts stripped before TF-IDF
- Entity extraction via curated regex dictionary (33 patterns: companies, products, people, platforms)
- All 11 events have genuine entities

In [ ]:
topics_df = pd.read_csv(R3 / 'data/processed/topic_entity_by_event.csv')

has_entities = topics_df['current_entities'].notna() & (topics_df['current_entities'] != '')
print(f'Events with entities: {has_entities.sum()} / {len(topics_df)}')
print()
display(topics_df[['event_id', 'event_type', 'timestamp',
                   'current_topics', 'current_entities']].to_string(index=False))

## 7. Figure 2 — Event Topic Comparison (EVT_003)

In [ ]:
fig2 = mpimg.imread(str(R3 / 'reports/figure2_topics.png'))
plt.figure(figsize=(10, 5))
plt.imshow(fig2)
plt.axis('off')
plt.title('Figure 2: Event Topic Analysis', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Trigger Investigation

Built by `Round3/src/06_trigger_lookup.py`.
- Source: HackerNews Algolia API (unauthenticated)
- Search window: 24h before to 6h after each event
- 79 candidate articles retrieved across 11 events

In [ ]:
triggers_df = pd.read_csv(R3 / 'data/processed/trigger_candidates.csv')

print(f'Total candidate articles fetched: {len(triggers_df)}')
print(f'Events covered: {triggers_df["event_id"].nunique()}')
print()
print('Final classification: 0 Strong / 1 Weak / 10 Unexplained')
print('Weak: EVT_003 — iPhone 18 teardown video, published ~20h before event')
print('All others: Unexplained (HN API too developer-oriented for consumer news triggers)')

## 9. Limitations

1. **Single-shot collection** — no continuous monitoring baseline
2. **Bluesky scope** — smaller than Twitter/Reddit
3. **6 future-dated `created_at` anomalies** — cause unknown, preserved unaltered
4. **Frozen Round 2 model** (Macro-F1 ~60%) — English-centric, struggles with cross-language/mixed-entity posts
5. **Entity extraction** — regex dictionary only; NLTK/spaCy unavailable in this environment
6. **10/11 events unexplained** — HackerNews API misses consumer-facing news triggers
7. **~21-hour observation window** — no pre-event baseline